<a href="https://colab.research.google.com/github/rpaulos/CCMACLRL_EXERCISES_COM232/blob/main/Exercise6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exercise 6: Choosing the best performing model on a dataset

Instructions:

- Use the Dataset File to train your model
- Use the Test File to generate your results
- Use the Sample Submission file to generate the same format
- Use all Regression models

Submit your results to:
https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview



In [115]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from matplotlib import pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn import datasets
from sklearn import svm
from sklearn.metrics import root_mean_squared_error

## Dataset File

In [88]:
train_data = 'https://github.com/robitussin/CCMACLRL_EXERCISES/blob/3fd7d51ffd17863598ac3f44eeefc558171a5b73/dataset/house-prices-advanced-regression-techniques/train.csv?raw=true'
df = pd.read_csv(train_data)

## Test File

In [89]:
test_url = 'https://github.com/robitussin/CCMACLRL_EXERCISES/blob/3fd7d51ffd17863598ac3f44eeefc558171a5b73/dataset/house-prices-advanced-regression-techniques/test.csv?raw=true'
dt=pd.read_csv(test_url)

In [90]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   object 
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   object 
 6   Alley          107 non-null    object 
 7   LotShape       1459 non-null   object 
 8   LandContour    1459 non-null   object 
 9   Utilities      1457 non-null   object 
 10  LotConfig      1459 non-null   object 
 11  LandSlope      1459 non-null   object 
 12  Neighborhood   1459 non-null   object 
 13  Condition1     1459 non-null   object 
 14  Condition2     1459 non-null   object 
 15  BldgType       1459 non-null   object 
 16  HouseStyle     1459 non-null   object 
 17  OverallQual    1459 non-null   int64  
 18  OverallC

In [91]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [92]:
df.tail()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125
1459,1460,20,RL,75.0,9937,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2008,WD,Normal,147500


In [93]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    "MSZoning","Street","LotShape","LandContour","Utilities","LotConfig",
    "LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle",
    "RoofStyle","RoofMatl","Exterior1st","Exterior2nd","MasVnrType","ExterQual",
    "ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1",
    "BsmtFinType2","Heating","HeatingQC","CentralAir","Electrical","KitchenQual",
    "Functional","FireplaceQu","GarageType","GarageFinish","GarageQual","GarageCond",
    "PavedDrive","PoolQC","Fence","MiscFeature","SaleType","SaleCondition"
]

le = LabelEncoder()

for col in cat_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])


In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   int64  
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   int64  
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   int64  
 8   LandContour    1460 non-null   int64  
 9   Utilities      1460 non-null   int64  
 10  LotConfig      1460 non-null   int64  
 11  LandSlope      1460 non-null   int64  
 12  Neighborhood   1460 non-null   int64  
 13  Condition1     1460 non-null   int64  
 14  Condition2     1460 non-null   int64  
 15  BldgType       1460 non-null   int64  
 16  HouseStyle     1460 non-null   int64  
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [95]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,3,65.0,8450,1,NaN,3,3,0,...,0,3,4,4,0,2,2008,8,4,208500
1,2,20,3,80.0,9600,1,NaN,3,3,0,...,0,3,4,4,0,5,2007,8,4,181500
2,3,60,3,68.0,11250,1,NaN,0,3,0,...,0,3,4,4,0,9,2008,8,4,223500
3,4,70,3,60.0,9550,1,NaN,0,3,0,...,0,3,4,4,0,2,2006,8,0,140000
4,5,60,3,84.0,14260,1,NaN,0,3,0,...,0,3,4,4,0,12,2008,8,4,250000


In [96]:
df.isna().sum()

na = df.isna().sum().reset_index()
na.columns = ['Column', 'MissingValues']
na

,Column,MissingValues
0,Id,0
1,MSSubClass,0
2,MSZoning,0
3,LotFrontage,259
4,LotArea,0
...,...,...
76,MoSold,0
77,YrSold,0
78,SaleType,0
79,SaleCondition,0


In [97]:
# 1. Drop rows where LotFrontage has missing values
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

# 2. Drop the Alley column
df = df.drop(columns=['Alley'])

# 3. Fill empty values in MasVnrArea with its median
df['MasVnrArea'] = df['MasVnrArea'].fillna(df['MasVnrArea'].median())

# 4. Fill empty values in GarageYrBlt with its median
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['GarageYrBlt'].median())

In [98]:
missing = df.isnull().sum().reset_index()
missing.columns = ['Column', 'MissingValues']
missing

,Column,MissingValues
0,Id,0
1,MSSubClass,0
2,MSZoning,0
3,LotFrontage,0
4,LotArea,0
...,...,...
75,MoSold,0
76,YrSold,0
77,SaleType,0
78,SaleCondition,0


## Sample Submission File

In [99]:
sample_submission_url ='https://github.com/robitussin/CCMACLRL_EXERCISES/blob/3fd7d51ffd17863598ac3f44eeefc558171a5b73/dataset/house-prices-advanced-regression-techniques/sample_submission.csv?raw=true'

sf=pd.read_csv(sample_submission_url)

In [100]:
sf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Id         1459 non-null   int64  
 1   SalePrice  1459 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 22.9 KB


# Split Dataset Test and Train

In [101]:
y = df['SalePrice']
X = df.drop(['SalePrice'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=0
)

In [102]:
score_list = {}

## 1. Train a KNN Regressor

In [103]:
# put your answer here
from sklearn.neighbors import KNeighborsRegressor
KNN = KNeighborsRegressor(n_neighbors=22) #I've tried more than 50 values. 22 is the best value

KNN.fit(x_train,y_train)
knn_score = KNN.score(x_test,y_test)
score_list["KNN Classifier"] = knn_score
print(f"Score is {knn_score}")

Score is 0.5959593546886721


- Perform cross validation

In [116]:
# put your answer here
y_pred_knn = KNN.predict(x_test)
rmse_knn = root_mean_squared_error(y_test, y_pred_knn)
print(f"RMSE is {rmse_knn}")

RMSE is 50170.41689470754


## 2. Train a SVM Regression

In [105]:
# put your answer here
from sklearn.svm import SVC

svc = SVC()
svc.fit(x_train,y_train)
svc_score = svc.score(x_test,y_test)
score_list["SVC"] = svc_score
print(f"Score is {svc_score}")

Score is 0.0136986301369863


In [117]:
y_pred_svc = svc.predict(x_test)
rmse_svc = root_mean_squared_error(y_test, y_pred_svc)
print(f"RMSE is {rmse_svc}")

RMSE is 86813.8338567804


## 3. Train a Decision Tree Regression

In [107]:
# put your answer here
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier(random_state=1)
dtc.fit(x_train,y_train)

dtc_score = dtc.score(x_test,y_test)
score_list["DTC"] = dtc_score
print(f"Score is {dtc_score}")

Score is 0.008561643835616438


- Perform cross validation

In [118]:
# put your answer here
y_pred_dtc = dtc.predict(x_test)
rmse_dtc = root_mean_squared_error(y_test, y_pred_dtc)
print(f"RMSE is {rmse_dtc}")

RMSE is 55561.49444992063


## 4. Train a Random Forest Regression

In [109]:
# put your answer here
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=50,random_state=1)
rfc.fit(x_train,y_train)
rfc_score = rfc.score(x_test,y_test)
score_list["RFC"]=rfc_score

print(f"Score is {rfc_score}")

Score is 0.0136986301369863


## 5. Compare all the performance of all regression models

In [110]:
# put your answer here
from pprint import pprint
pprint(score_list)

{'DTC': 0.008561643835616438,
 'KNN Classifier': 0.5959593546886721,
 'RFC': 0.0136986301369863,
 'SVC': 0.0136986301369863}


## 6. Generate Submission File

Choose the model that has the best performance to generate a submission file.

## Data Cleaning For SUbmission

In [111]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    "MSZoning","Street","LotShape","LandContour","Utilities","LotConfig",
    "LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle",
    "RoofStyle","RoofMatl","Exterior1st","Exterior2nd","MasVnrType","ExterQual",
    "ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1",
    "BsmtFinType2","Heating","HeatingQC","CentralAir","Electrical","KitchenQual",
    "Functional","FireplaceQu","GarageType","GarageFinish","GarageQual","GarageCond",
    "PavedDrive","PoolQC","Fence","MiscFeature","SaleType","SaleCondition"
]

le = LabelEncoder()

for col in cat_cols:
    dt[col] = dt[col].astype(str)
    dt[col] = le.fit_transform(dt[col])

In [112]:
# 1. Drop rows where LotFrontage has missing values
dt['LotFrontage'] = dt['LotFrontage'].fillna(dt['LotFrontage'].median())

# 2. Drop the Alley column
dt = dt.drop(columns=['Alley'])

# 3. Fill empty values in MasVnrArea with its median
dt['MasVnrArea'] = dt['MasVnrArea'].fillna(dt['MasVnrArea'].median())

# 4. Fill empty values in GarageYrBlt with its median
dt['GarageYrBlt'] = dt['GarageYrBlt'].fillna(dt['GarageYrBlt'].median())

In [113]:
id = sf.pop('Id')
y_pred = rfc.predict(dt)

# Create a submission DataFrame
submission_df = pd.DataFrame({
    'Id': id,
    'SalePrice': y_pred
})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)
print("Submission file created: submission_file.csv")

Submission file created: submission_file.csv
